[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_76_Phase8_Capstone_Shipping_AgentObs.ipynb)

# Lesson 76 - Phase 8 Capstone: Shipping `agent-obs` 🚀

**Where we are.** Over Lessons 71-75 you built six observability tools, and you
validated each one *in isolation* inside its own notebook:

| Lesson | Module | What it does |
|---|---|---|
| 71 | `tracing` + `online_eval` | spans, metrics, label-free quality + drift |
| 72 | `logging_search` | structured logs, context propagation, `TraceStore` |
| 73 | `alerting` | SLOs, error budgets, burn-rate alerts |
| 74 | `rollout` | sticky A/B buckets + guarded staged rollout |
| 75 | `feedback` | signals -> labeled data -> data flywheel |

**Today (the capstone).** Six modules that *each* pass their own tests are **not**
a library. This lesson turns them into one installable package - `agent-obs` -
with a single public API, proves the six compose by running them **together** in
one kernel against a benchmark, then ships it launch-day style as your **third**
public artifact (after `paper-distiller` and `agent-bench`).

> **The load-bearing risk of a capstone:** modules that were only ever tested
> apart can silently diverge - a renamed field here, a changed return shape there -
> and nobody notices until they're wired together. The integration test below is
> the whole point of this lesson.

*No API key needed. Everything runs deterministically on a mock agent.*

In [ ]:
# --- Setup: deterministic, no API key ---------------------------------------
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("rich>=13", "build", "twine")

import os, json, random, shutil
from pathlib import Path

# BASE = "/content" on Colab. One switch controls every path in the notebook.
BASE = "/content"
PKG_DIR = os.path.join(BASE, "agent-obs")           # the package repo we build
OBS_DIR = os.path.join(PKG_DIR, "observability")    # the import package
RUN_DIR = os.path.join(BASE, "obs_run")             # scratch for the integration run
for d in (OBS_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)

SEED = 76
random.seed(SEED)
print("setup ok - BASE =", BASE)
print("package dir:", PKG_DIR)

## 1. From six notebooks to one package

Right now the modules live as strings scattered across five notebooks. A user
can't `pip install` a notebook. Consolidation means three concrete things:

1. **One import surface.** A single `observability/__init__.py` re-exports the
   whole public API, so a user writes `from observability import Tracer, decide,
   GoldenSet` instead of hunting through submodules.
2. **One distribution.** A `pyproject.toml` describes how to build a wheel. Note
   the deliberate split you learned in Lesson 58/68: the **distribution name**
   (`agent-obs`, what you `pip install`) differs from the **import name**
   (`observability`, what you `import`). Both are legal; keep them intentional.
3. **One version.** `__version__ = "0.1.0"` in one place.

We also have to resolve two real name collisions the six modules didn't have to
worry about while they lived apart: both `logging_search` and `feedback` define a
`redact`, and both `alerting` and `rollout` define a `burn_rate`. The `__init__`
picks one of each for the top-level namespace (the log-path `redact` and the SLO
`burn_rate`); the other stays reachable via its submodule. **This is exactly the
kind of divergence a capstone surfaces.**

In [ ]:
# --- Consolidate: write the package to disk -------------------------------
_SRC = {}
_SRC["tracing.py"] = r"""
# observability/tracing.py — portable span tracer (OTel GenAI-style attribute names).
import time, uuid, json, contextlib
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class Span:
    name: str; trace_id: str; span_id: str; parent_id: Optional[str]
    start: float; end: Optional[float] = None; status: str = "ok"
    attrs: dict = field(default_factory=dict)
    @property
    def latency_ms(self): return round(((self.end or self.start) - self.start) * 1000, 2)

class Tracer:
    # One JSONL line per finished span; nests via an internal stack for parent linking.
    def __init__(self, sink_path): self.sink_path = sink_path; self._stack = []
    @contextlib.contextmanager
    def span(self, name, trace_id=None, **attrs):
        parent = self._stack[-1] if self._stack else None
        s = Span(name, trace_id or (parent.trace_id if parent else uuid.uuid4().hex[:12]),
                 uuid.uuid4().hex[:12], parent.span_id if parent else None,
                 time.perf_counter(), attrs=dict(attrs))
        self._stack.append(s)
        try:
            yield s
        except Exception as e:
            s.status = "error"; s.attrs["error.type"] = type(e).__name__; raise
        finally:
            s.end = time.perf_counter(); self._stack.pop()
            rec = asdict(s); rec["latency_ms"] = s.latency_ms
            with open(self.sink_path, "a") as f: f.write(json.dumps(rec) + "\n")
"""
_SRC["online_eval.py"] = r"""
# observability/online_eval.py — label-free online quality signals + drift monitor.
import statistics

def heuristic_score(resp: dict) -> dict:
    # Cheap proxies you can run on 100% of production traffic (no labels needed).
    if "error" in resp: return {"score": 0.0, "reasons": ["request_errored"]}
    score, reasons = 1.0, []
    if resp.get("n_tokens_out", 0) < 60: score -= 0.5; reasons.append("output_too_short")
    if not resp.get("key_points"):       score -= 0.4; reasons.append("no_key_points")
    return {"score": round(max(0.0, score), 3), "reasons": reasons or ["ok"]}

def sample_indices(n, rate=0.30, seed=71):
    import random; rng = random.Random(seed)
    return [i for i in range(n) if rng.random() < rate]

class DriftMonitor:
    # Rolling mean of a quality signal vs a fixed offline baseline; alarm on the gap.
    def __init__(self, baseline, window=20, alarm_drop=0.20):
        self.baseline, self.window, self.alarm_drop = baseline, window, alarm_drop
        self.buf = []
    def update(self, score) -> bool:
        self.buf.append(score); self.buf = self.buf[-self.window:]
        return statistics.mean(self.buf) < self.baseline - self.alarm_drop
"""
_SRC["logging_search.py"] = r"""
'''observability.logging_search - structured logging, context propagation,
redaction, and a searchable TraceStore. Built in Lesson 72.'''
import json, uuid, re, contextvars
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from contextlib import contextmanager

LEVELS = {"DEBUG": 10, "INFO": 20, "WARNING": 30, "ERROR": 40}
_current_ctx = contextvars.ContextVar("request_ctx", default=None)

@dataclass
class RequestContext:
    trace_id: str; session_id: str; user_id: str; request_id: str

def current_context():
    c = _current_ctx.get()
    return asdict(c) if c else {}

@contextmanager
def request_scope(user_id, session_id, request_id=None, trace_id=None):
    ctx = RequestContext(
        trace_id=trace_id or "trace_" + uuid.uuid4().hex[:12],
        session_id=session_id, user_id=user_id,
        request_id=request_id or "req_" + uuid.uuid4().hex[:8])
    token = _current_ctx.set(ctx)
    try:
        yield ctx
    finally:
        _current_ctx.reset(token)

DENY_KEYS = {"api_key", "password", "authorization", "secret", "token"}
PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9\-]{6,}"), "[REDACTED_KEY]"),
    (re.compile(r"Bearer\s+[A-Za-z0-9\.\-_]+"), "Bearer [REDACTED]"),
    (re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"), "[REDACTED_EMAIL]"),
]

def redact(record):
    clean = {}
    for k, v in record.items():
        if k in DENY_KEYS:
            clean[k] = "[REDACTED]"; continue
        if isinstance(v, str):
            for pat, repl in PATTERNS:
                v = pat.sub(repl, v)
        clean[k] = v
    return clean

class StructuredLogger:
    def __init__(self, sink, min_level="DEBUG", clock=None,
                 context_provider=None, do_redact=True):
        self.sink = sink; self.min_level = LEVELS[min_level]
        self.clock = clock or (lambda: datetime.now(timezone.utc))
        self._context_provider = context_provider or (lambda: {})
        self.do_redact = do_redact
    def _emit(self, level, event, **fields):
        if LEVELS[level] < self.min_level:
            return
        record = {"ts": self.clock().isoformat(), "level": level, "event": event,
                  **self._context_provider(), **fields}
        if self.do_redact:
            record = redact(record)
        self.sink.write(json.dumps(record) + "\n"); self.sink.flush()
        return record
    def debug(self, e, **f):   return self._emit("DEBUG", e, **f)
    def info(self, e, **f):    return self._emit("INFO", e, **f)
    def warning(self, e, **f): return self._emit("WARNING", e, **f)
    def error(self, e, **f):   return self._emit("ERROR", e, **f)

class Query:
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __iter__(self): return iter(self.rows)
    def user(self, u):    return Query([r for r in self.rows if r.get("user_id") == u])
    def session(self, s): return Query([r for r in self.rows if r.get("session_id") == s])
    def trace(self, t):   return Query([r for r in self.rows if r.get("trace_id") == t])
    def event(self, n):   return Query([r for r in self.rows if r.get("event") == n])
    def failed(self):     return Query([r for r in self.rows
                          if r.get("status") == "ERROR" or r.get("level") == "ERROR"])
    def slower_than(self, ms):
        return Query([r for r in self.rows if (r.get("latency_ms") or 0) > ms])
    def search_text(self, needle):
        n = needle.lower()
        return Query([r for r in self.rows if n in json.dumps(r).lower()])
    def since(self, dt):
        return Query([r for r in self.rows
                      if r.get("ts") and datetime.fromisoformat(r["ts"]) >= dt])
    def last(self, minutes, now):
        return self.since(now - timedelta(minutes=minutes))
    def sort_by(self, key, reverse=True):
        return Query(sorted(self.rows, key=lambda r: r.get(key) or 0, reverse=reverse))

class TraceStore:
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path) if l.strip()]
    def all(self):   return Query(self.rows)
    def logs(self):  return Query([r for r in self.rows if "level" in r])
    def spans(self): return Query([r for r in self.rows if r.get("kind") == "span"])
"""
_SRC["alerting.py"] = r"""
'''
observability/alerting.py

Reusable SLO + alerting engine for LLM/agent systems.
Builds on the metrics (Lesson 71) and structured logs / trace search (Lesson 72)
you already emit. Nothing here needs an API key; it operates on events you log.

Core objects:
  SLO            - a reliability promise (target success rate over a window)
  error_budget   - how much failure the SLO allows, and how much is left
  burn_rate      - how fast you are spending that budget right now
  AlertRule      - a flap-resistant state machine (OK -> PENDING -> FIRING -> OK)
                   with a for-duration and hysteresis (separate recover threshold)
  Severity/route - decide whether a condition PAGES a human, files a TICKET, or LOGS
'''
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional


# ---- SLI / SLO / error budget -------------------------------------------------

@dataclass
class SLO:
    # A Service Level Objective: the promise. Example: 99% of requests succeed
    # over a rolling 30-day window. We keep the window in minutes for the demo.
    name: str
    target: float          # e.g. 0.99  (the SLI we promise to stay at or above)
    window_minutes: int    # the compliance window the target is measured over

    @property
    def allowed_error_rate(self) -> float:
        # The error budget expressed as a rate. SLO 99% -> 1% of requests may fail.
        return 1.0 - self.target


def sli_success_rate(n_total: int, n_bad: int) -> float:
    # A Service Level Indicator: a measured number. Here, fraction of good events.
    if n_total <= 0:
        return 1.0
    return 1.0 - (n_bad / n_total)


def error_budget(slo: SLO, n_total: int, n_bad: int) -> dict:
    # Budget = allowed failures over the window. Spent = actual failures.
    # Returned 'remaining_frac' < 0 means the SLO is already blown for this window.
    budget = slo.allowed_error_rate * n_total
    spent = float(n_bad)
    remaining = budget - spent
    remaining_frac = (remaining / budget) if budget > 0 else 0.0
    return {
        'budget_events': budget,
        'spent_events': spent,
        'remaining_events': remaining,
        'remaining_frac': remaining_frac,
        'sli': sli_success_rate(n_total, n_bad),
        'slo_met': sli_success_rate(n_total, n_bad) >= slo.target,
    }


def burn_rate(slo: SLO, window_error_rate: float) -> float:
    # Burn rate = how many times faster than 'sustainable' you are spending budget.
    # 1.0 = spending exactly at the pace that would exhaust the budget over the
    # full window. 14.4 over 1h = a whole 30-day budget gone in ~2 days.
    allowed = slo.allowed_error_rate
    if allowed <= 0:
        return float('inf') if window_error_rate > 0 else 0.0
    return window_error_rate / allowed


# ---- Alert state machine ------------------------------------------------------

class AlertState(Enum):
    OK = 'ok'            # condition not met
    PENDING = 'pending'  # condition met, but not yet for long enough to page
    FIRING = 'firing'    # condition sustained past for_duration -> alert is live


@dataclass
class AlertEvent:
    minute: int
    old_state: AlertState
    new_state: AlertState
    value: float


@dataclass
class AlertRule:
    # A flap-resistant rule. Two design ideas that separate a real alert from a
    # naive threshold check:
    #   for_duration     - the condition must hold this many steps before FIRING
    #                       (kills single-sample spikes)
    #   recover_threshold - a SEPARATE, lower threshold to leave FIRING (hysteresis)
    #                       so a value hovering at the line does not flap on/off
    name: str
    fire_threshold: float
    recover_threshold: float
    for_duration: int = 3
    recover_duration: int = 3
    severity: str = 'ticket'

    state: AlertState = field(default=AlertState.OK, init=False)
    _above_count: int = field(default=0, init=False)
    _below_count: int = field(default=0, init=False)

    def update(self, minute: int, value: float) -> Optional[AlertEvent]:
        # Feed one measurement. Returns an AlertEvent iff the state changed.
        old = self.state
        if self.state in (AlertState.OK, AlertState.PENDING):
            if value >= self.fire_threshold:
                self._above_count += 1
                self._below_count = 0
                if self.state == AlertState.OK:
                    self.state = AlertState.PENDING
                if self._above_count >= self.for_duration:
                    self.state = AlertState.FIRING
            else:
                self._above_count = 0
                self.state = AlertState.OK
        elif self.state == AlertState.FIRING:
            # Hysteresis: only leave FIRING once we drop under the LOWER
            # recover_threshold for recover_duration steps.
            if value < self.recover_threshold:
                self._below_count += 1
                if self._below_count >= self.recover_duration:
                    self.state = AlertState.OK
                    self._above_count = 0
                    self._below_count = 0
            else:
                self._below_count = 0

        if self.state != old:
            return AlertEvent(minute, old, self.state, value)
        return None


# ---- Severity routing ---------------------------------------------------------

def route(severity: str) -> str:
    # Where does a firing alert go? The whole point of severity is to protect
    # human attention: only real, urgent, actionable problems PAGE.
    table = {
        'page': 'PAGE on-call (wake a human now)',
        'ticket': 'FILE ticket (handle in business hours)',
        'log': 'LOG only (record, do not notify)',
    }
    return table.get(severity, 'LOG only (record, do not notify)')


def severity_for_burn(fast_burn: float, slow_burn: float) -> str:
    # Multi-window, multi-burn-rate (Google SRE). A fast, high burn on a short
    # window PAGES; a slower burn confirmed on a long window is a TICKET; small
    # burns just LOG. Requiring the long window too suppresses one-off blips.
    if fast_burn >= 14.4 and slow_burn >= 6.0:
        return 'page'
    if fast_burn >= 3.0 and slow_burn >= 1.0:
        return 'ticket'
    return 'log'
"""
_SRC["rollout.py"] = r"""
# observability/rollout.py
# Guarded-rollout toolkit: sticky bucketing, two-proportion significance test,
# burn-rate guardrail, and a staged-ramp controller. Zero third-party deps.
import math, hashlib
from dataclasses import dataclass
from typing import Optional


def bucket_of(unit_id, salt="exp"):
    # stable hash of unit_id into [0, 1)
    h = hashlib.md5((salt + ":" + str(unit_id)).encode()).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


def assign_arm(unit_id, split, salt="exp"):
    # "B" (challenger) if the unit falls under the split, else "A" (champion)
    return "B" if bucket_of(unit_id, salt) < split else "A"


def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


@dataclass
class TestResult:
    p_a: float
    p_b: float
    diff: float
    z: float
    p_value: float
    ci_low: float
    ci_high: float
    n_a: int
    n_b: int


def two_proportion_ztest(succ_a, n_a, succ_b, n_b, z_crit=1.96):
    p_a = succ_a / n_a
    p_b = succ_b / n_b
    p_pool = (succ_a + succ_b) / (n_a + n_b)
    se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se_pool if se_pool > 0 else 0.0
    p_value = 2 * (1 - normal_cdf(abs(z)))
    se_diff = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    diff = p_b - p_a
    return TestResult(p_a, p_b, diff, z, p_value,
                      diff - z_crit * se_diff, diff + z_crit * se_diff, n_a, n_b)


def error_budget_allowed(slo):
    return 1.0 - slo


def burn_rate(observed_error_rate, slo):
    allowed = error_budget_allowed(slo)
    if allowed <= 0:
        return float("inf")
    return observed_error_rate / allowed


@dataclass
class GuardrailConfig:
    slo: float = 0.95
    fast_burn: float = 14.4
    min_samples: int = 150


def guardrail_breached(succ_b, n_b, cfg):
    if n_b < cfg.min_samples:
        return False, 0.0
    err = 1.0 - succ_b / n_b
    br = burn_rate(err, cfg.slo)
    return br >= cfg.fast_burn, br


@dataclass
class Decision:
    action: str      # PROMOTE | HOLD | ROLLBACK
    reason: str
    split: float
    burn_b: float
    p_value: float


def decide(succ_a, n_a, succ_b, n_b, split, cfg=None):
    # One rollout decision from this stage's counts. Guardrail first (safety),
    # then the significance test (decision).
    cfg = cfg or GuardrailConfig()
    breached, burn = guardrail_breached(succ_b, n_b, cfg)
    if n_a > 30 and n_b > 30:
        tr = two_proportion_ztest(succ_a, n_a, succ_b, n_b)
        pval, diff = tr.p_value, tr.diff
    else:
        pval, diff = 1.0, 0.0
    if breached:
        return Decision("ROLLBACK", "burn-rate meltdown", split, burn, pval)
    if n_b > 30 and pval < 0.05 and diff < 0:
        return Decision("ROLLBACK", "significantly worse", split, burn, pval)
    if n_b < cfg.min_samples:
        return Decision("HOLD", "insufficient samples", split, burn, pval)
    return Decision("PROMOTE", "healthy", split, burn, pval)


DEFAULT_RAMP = [0.05, 0.25, 0.50, 1.0]
"""
_SRC["feedback.py"] = r"""
# observability/feedback.py
# Lesson 75 - Feedback loops & the data flywheel.
# Turn production signal into labeled data, and labeled data into a better eval set.

from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# ---------------------------------------------------------------- signals

# Weight of each implicit/explicit signal as evidence that a response was BAD.
# Positive weight = evidence of badness. Negative weight = evidence of goodness.
DEFAULT_WEIGHTS: Dict[str, float] = {
    "thumbs_down": 3.0,
    "thumbs_up": -3.0,
    "regenerated": 1.5,
    "edited": 1.0,
    "abandoned": 1.2,
    "guardrail_fired": 2.5,
    "low_heuristic": 1.0,
    "low_judge": 2.0,
    "high_judge": -1.5,
}


@dataclass
class FeedbackEvent:
    # One production request plus every signal we observed about it.
    req_id: str
    user_id: str
    topic: str
    text: str = ""
    heuristic_score: float = 1.0
    judge_score: Optional[float] = None
    thumbs: Optional[int] = None          # +1 / -1 / None (explicit, rare)
    regenerated: bool = False             # implicit: user hit retry
    edited: bool = False                  # implicit: user rewrote the answer
    abandoned: bool = False               # implicit: user left the session
    guardrail_fired: bool = False         # from L73 alerting
    arm: str = "champion"                 # from L74 rollout
    meta: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def active_signals(ev: FeedbackEvent,
                   heuristic_floor: float = 0.55,
                   judge_floor: float = 0.5,
                   judge_ceiling: float = 0.85) -> List[str]:
    # Which named signals are present on this event.
    out: List[str] = []
    if ev.thumbs == -1:
        out.append("thumbs_down")
    if ev.thumbs == 1:
        out.append("thumbs_up")
    if ev.regenerated:
        out.append("regenerated")
    if ev.edited:
        out.append("edited")
    if ev.abandoned:
        out.append("abandoned")
    if ev.guardrail_fired:
        out.append("guardrail_fired")
    if ev.heuristic_score < heuristic_floor:
        out.append("low_heuristic")
    if ev.judge_score is not None and ev.judge_score < judge_floor:
        out.append("low_judge")
    if ev.judge_score is not None and ev.judge_score >= judge_ceiling:
        out.append("high_judge")
    return out


def _logistic(x: float) -> float:
    import math
    return 1.0 / (1.0 + math.exp(-x))


@dataclass
class WeakLabel:
    # A guess at the true label, plus how confident we are and why.
    label: str            # "bad" | "good" | "unknown"
    p_bad: float          # calibrated-ish probability the response was bad
    signals: List[str]
    def is_confident(self, lo: float = 0.25, hi: float = 0.75) -> bool:
        return self.p_bad <= lo or self.p_bad >= hi


def weak_label(ev: FeedbackEvent,
               weights: Optional[Dict[str, float]] = None,
               lo: float = 0.25,
               hi: float = 0.75) -> WeakLabel:
    # Fuse noisy signals into ONE weak label. This is a prior, never ground truth.
    w = weights or DEFAULT_WEIGHTS
    sigs = active_signals(ev)
    score = sum(w.get(s, 0.0) for s in sigs)
    p_bad = _logistic(score - 1.0)   # -1.0 bias: silence means probably fine
    if p_bad >= hi:
        lab = "bad"
    elif p_bad <= lo:
        lab = "good"
    else:
        lab = "unknown"
    return WeakLabel(label=lab, p_bad=p_bad, signals=sigs)


# ---------------------------------------------------------------- sampling

def uncertainty(ev: FeedbackEvent, wl: Optional[WeakLabel] = None) -> float:
    # Higher = more informative to send to a human. Two ingredients:
    #   1. label uncertainty  (p_bad near 0.5)
    #   2. scorer disagreement (heuristic and judge tell different stories)
    wl = wl or weak_label(ev)
    label_unc = 1.0 - 2.0 * abs(wl.p_bad - 0.5)          # 0..1, peaks at p=0.5
    if ev.judge_score is None:
        disagree = 0.0
    else:
        disagree = abs(ev.heuristic_score - ev.judge_score)
    return 0.6 * label_unc + 0.4 * disagree


def sample_for_review(events: Sequence[FeedbackEvent],
                      budget: int,
                      strategy: str = "uncertainty",
                      seed: int = 0) -> List[FeedbackEvent]:
    # Choose which `budget` events a human will actually label.
    if strategy == "random":
        import random
        rng = random.Random(seed)
        pool = list(events)
        rng.shuffle(pool)
        return pool[:budget]
    if strategy == "uncertainty":
        ranked = sorted(events, key=lambda e: -uncertainty(e))
        return list(ranked[:budget])
    if strategy == "confident_bad":
        ranked = sorted(events, key=lambda e: -weak_label(e).p_bad)
        return list(ranked[:budget])
    raise ValueError("unknown strategy: " + strategy)


def stratified_sample(events: Sequence[FeedbackEvent],
                      budget: int,
                      key=lambda e: e.topic,
                      seed: int = 0) -> List[FeedbackEvent]:
    # Even coverage across strata, so a loud topic cannot eat the whole budget.
    import random
    rng = random.Random(seed)
    buckets: Dict[Any, List[FeedbackEvent]] = {}
    for e in events:
        buckets.setdefault(key(e), []).append(e)
    for v in buckets.values():
        rng.shuffle(v)
    out: List[FeedbackEvent] = []
    keys = sorted(buckets, key=str)
    i = 0
    while len(out) < budget and any(buckets[k] for k in keys):
        k = keys[i % len(keys)]
        if buckets[k]:
            out.append(buckets[k].pop())
        i += 1
    return out


# ---------------------------------------------------------------- dataset

_PII_PATTERNS = [
    re.compile(r"[\w\.\-\+]+@[\w\-]+\.[\w\.\-]+"),
    re.compile(r"sk-[A-Za-z0-9\-_]{6,}"),
    re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
]


def redact(text: str) -> str:
    for pat in _PII_PATTERNS:
        text = pat.sub("[REDACTED]", text)
    return text


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


def content_hash(text: str) -> str:
    return hashlib.md5(normalize(text).encode("utf-8")).hexdigest()[:16]


@dataclass
class EvalCase:
    case_id: str
    prompt: str
    topic: str
    label: str            # "bad" | "good"
    source: str           # "production" | "seed"
    provenance: Dict[str, Any] = field(default_factory=dict)


class GoldenSet:
    # The curated eval set that agent-bench runs against.
    def __init__(self, cases: Optional[Iterable[EvalCase]] = None):
        self.cases: List[EvalCase] = list(cases or [])
        self._hashes = {content_hash(c.prompt) for c in self.cases}

    def __len__(self) -> int:
        return len(self.cases)

    def contains(self, prompt: str) -> bool:
        return content_hash(prompt) in self._hashes

    def add(self, case: EvalCase) -> bool:
        # Returns False if rejected as a duplicate / already-present case.
        h = content_hash(case.prompt)
        if h in self._hashes:
            return False
        self._hashes.add(h)
        self.cases.append(case)
        return True

    def topics(self) -> Dict[str, int]:
        out: Dict[str, int] = {}
        for c in self.cases:
            out[c.topic] = out.get(c.topic, 0) + 1
        return out


def build_cases(reviewed: Sequence[Tuple[FeedbackEvent, str]],
                golden: GoldenSet,
                source: str = "production") -> Tuple[List[EvalCase], Dict[str, int]]:
    # reviewed = [(event, human_label)]. Redact, dedup within batch, and refuse
    # anything already in the golden set (test-set contamination guard).
    accepted: List[EvalCase] = []
    stats = {"seen": 0, "dup_in_batch": 0, "already_in_golden": 0, "accepted": 0}
    seen_batch = set()
    for ev, human in reviewed:
        stats["seen"] += 1
        prompt = redact(ev.text)
        h = content_hash(prompt)
        if golden.contains(prompt):
            stats["already_in_golden"] += 1
            continue
        if h in seen_batch:
            stats["dup_in_batch"] += 1
            continue
        seen_batch.add(h)
        accepted.append(EvalCase(
            case_id="case_" + h,
            prompt=prompt,
            topic=ev.topic,
            label=human,
            source=source,
            provenance={"req_id": ev.req_id, "signals": active_signals(ev), "arm": ev.arm},
        ))
        stats["accepted"] += 1
    return accepted, stats


# ---------------------------------------------------------------- flywheel

@dataclass
class FlywheelState:
    turn: int = 0
    golden_size: int = 0
    human_labels_spent: int = 0
    detected_regressions: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

    def record(self, **kw: Any) -> None:
        row = {"turn": self.turn, "golden_size": self.golden_size,
               "human_labels_spent": self.human_labels_spent}
        row.update(kw)
        self.history.append(row)
"""
_INIT = r'''
"""agent-obs - a production observability + online-evals toolkit for LLM agents.

Six composable pillars, each shippable on its own:
  tracing         - OTel-GenAI-style span tracer            (Lesson 71)
  online_eval     - label-free quality signals + drift      (Lesson 71)
  logging_search  - structured logs, context, TraceStore    (Lesson 72)
  alerting        - SLOs, error budgets, burn-rate alerts   (Lesson 73)
  rollout         - sticky A/B + guarded staged rollout      (Lesson 74)
  feedback        - signal -> labeled data -> data flywheel  (Lesson 75)
"""
from .tracing import Tracer, Span
from .online_eval import heuristic_score, sample_indices, DriftMonitor
from .logging_search import (RequestContext, current_context, request_scope,
                             redact, StructuredLogger, Query, TraceStore)
from .alerting import (SLO, sli_success_rate, error_budget, burn_rate,
                       AlertState, AlertEvent, AlertRule, route, severity_for_burn)
from .rollout import (bucket_of, assign_arm, normal_cdf, two_proportion_ztest,
                      GuardrailConfig, guardrail_breached, Decision, decide,
                      DEFAULT_RAMP)
from .feedback import (FeedbackEvent, active_signals, WeakLabel, weak_label,
                       uncertainty, sample_for_review, stratified_sample,
                       normalize, content_hash, EvalCase, GoldenSet,
                       build_cases, FlywheelState, DEFAULT_WEIGHTS)

__version__ = "0.1.0"
__all__ = [
    "Tracer", "Span", "heuristic_score", "sample_indices", "DriftMonitor",
    "RequestContext", "current_context", "request_scope", "redact",
    "StructuredLogger", "Query", "TraceStore",
    "SLO", "sli_success_rate", "error_budget", "burn_rate", "AlertState",
    "AlertEvent", "AlertRule", "route", "severity_for_burn",
    "bucket_of", "assign_arm", "normal_cdf", "two_proportion_ztest",
    "GuardrailConfig", "guardrail_breached", "Decision", "decide", "DEFAULT_RAMP",
    "FeedbackEvent", "active_signals", "WeakLabel", "weak_label", "uncertainty",
    "sample_for_review", "stratified_sample", "normalize", "content_hash",
    "EvalCase", "GoldenSet", "build_cases", "FlywheelState", "DEFAULT_WEIGHTS",
]
'''
_PYPROJECT = r'''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-obs"
version = "0.1.0"
description = "Production observability & online-evals toolkit for LLM agents."
readme = "README.md"
requires-python = ">=3.10"
license = { text = "MIT" }
authors = [{ name = "Gourav Khanijoe" }]
keywords = ["llm", "observability", "evals", "agents", "tracing", "slo"]
dependencies = []

[project.optional-dependencies]
rich = ["rich>=13.0"]

[project.urls]
Homepage = "https://github.com/gouravkhanijoe/agent-obs"
"Sibling: agent-bench" = "https://github.com/gouravkhanijoe/agent-bench"
"Sibling: paper-distiller" = "https://github.com/gouravkhanijoe/paper-distiller"

[tool.hatch.build.targets.wheel]
packages = ["observability"]
'''
_README = r"""
# agent-obs

[![CI](https://github.com/gouravkhanijoe/agent-obs/actions/workflows/ci.yml/badge.svg)](https://github.com/gouravkhanijoe/agent-obs/actions)
[![PyPI](https://img.shields.io/pypi/v/agent-obs.svg)](https://pypi.org/project/agent-obs/)
![Python](https://img.shields.io/badge/python-3.10+-blue.svg)

**Production observability & online-evals for LLM agents** - tracing, structured
logs, SLO alerting, guarded rollouts, and a data flywheel, in one dependency-free
package.

```python
from observability import Tracer, heuristic_score, SLO, AlertRule, decide, GoldenSet
```

## Six pillars
| Import | Gives you |
|---|---|
| `Tracer`, `Span` | OTel-GenAI-style spans -> JSONL |
| `heuristic_score`, `DriftMonitor` | label-free quality signals + drift alarm |
| `StructuredLogger`, `TraceStore`, `Query` | searchable structured logs |
| `SLO`, `error_budget`, `burn_rate`, `AlertRule` | reliability alerting |
| `assign_arm`, `two_proportion_ztest`, `decide` | guarded A/B rollouts |
| `FeedbackEvent`, `weak_label`, `GoldenSet` | production signal -> eval data |

## Part of a system
- **[agent-bench](https://github.com/gouravkhanijoe/agent-bench)** - the offline
  benchmark harness `agent-obs` instruments (one span per task attempt).
- **[paper-distiller](https://github.com/gouravkhanijoe/paper-distiller)** - a
  real agent you can watch in production with these tools.

## Install
```bash
pip install agent-obs
```

## License
MIT
"""

for _name, _body in _SRC.items():
    with open(os.path.join(OBS_DIR, _name), 'w') as _f:
        _f.write(_body.lstrip('\n') + '\n')
with open(os.path.join(OBS_DIR, '__init__.py'), 'w') as _f:
    _f.write(_INIT.lstrip('\n') + '\n')
with open(os.path.join(PKG_DIR, 'pyproject.toml'), 'w') as _f:
    _f.write(_PYPROJECT.lstrip('\n') + '\n')
with open(os.path.join(PKG_DIR, 'README.md'), 'w') as _f:
    _f.write(_README.lstrip('\n') + '\n')

written = sorted(os.listdir(OBS_DIR))
print('wrote observability/:', written)
assert set(written) == {'__init__.py', 'tracing.py', 'online_eval.py', 'logging_search.py', 'alerting.py', 'rollout.py', 'feedback.py'}
print('pyproject.toml:', os.path.exists(os.path.join(PKG_DIR, 'pyproject.toml')))

In [ ]:
# --- Sanity: every module parses & every promised symbol exists (pre-install) ---
import ast
public = {
    "tracing.py":        ["Tracer", "Span"],
    "online_eval.py":    ["heuristic_score", "DriftMonitor"],
    "logging_search.py": ["request_scope", "StructuredLogger", "TraceStore", "Query"],
    "alerting.py":       ["SLO", "AlertRule", "burn_rate", "severity_for_burn"],
    "rollout.py":        ["assign_arm", "two_proportion_ztest", "decide", "GuardrailConfig"],
    "feedback.py":       ["FeedbackEvent", "weak_label", "GoldenSet", "build_cases", "FlywheelState"],
}
for fname, names in public.items():
    tree = ast.parse(open(os.path.join(OBS_DIR, fname)).read())
    defined = {n.name for n in tree.body
               if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    missing = [n for n in names if n not in defined]
    assert not missing, f"{fname} missing {missing}"
    print(f"  {fname:20s} OK  ({len(defined)} top-level defs)")
print("\nall six modules parse and expose their promised symbols")

## 2. The load-bearing integration test

This is the cell the whole capstone exists for. We `pip install -e .`, import the
**installed** package (not the on-disk strings), and drive a small
`agent-bench`-shaped benchmark - a `MockAgent` answering scored `Task`s - through
**all six pillars in a single run**:

- **tracing** wraps every attempt in spans (one JSONL line each);
- **online_eval** scores every response with label-free heuristics and watches a
  `DriftMonitor`;
- **logging_search** emits one structured log per attempt inside a `request_scope`,
  then we query the store for failures by user;
- **alerting** feeds a rolling error rate into an `AlertRule` behind a 95% `SLO`;
- **rollout** buckets users into champion/challenger arms and asks `decide()`
  whether to promote or roll back;
- **feedback** turns the caught failures into weak labels and mines them into a
  `GoldenSet`, turning the flywheel.

If any module's shape had drifted, this cell would raise. `assert INTEGRATION_OK`
at the end is the green light.

In [ ]:
# --- Install the package (editable) and import it ---------------------------
import importlib
# 1) Demonstrate the real packaging path: an editable install (what a dev does).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-e", PKG_DIR, "-q"],
                   capture_output=True, text=True)
print("editable install:", "ok" if r.returncode == 0 else "FAILED")
if r.returncode != 0:
    print(r.stderr[-500:])

# 2) Put the package on THIS kernel's import path and import the consolidated API.
#    (On a freshly-booted kernel, importing straight from the tree is the reliable
#     way to pick up a just-written package; the editable install above is what
#     makes `import observability` work in a NEW shell / in CI.)
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)
for _m in list(sys.modules):
    if _m == "observability" or _m.startswith("observability."):
        del sys.modules[_m]
importlib.invalidate_caches()

import observability as obs
print("imported observability", obs.__version__, "from", os.path.dirname(obs.__file__))
print("public symbols exported:", len(obs.__all__))
assert obs.__version__ == "0.1.0"
assert len(obs.__all__) >= 40

In [ ]:
# --- ONE run, all six pillars ------------------------------------------------
from observability import (
    Tracer, heuristic_score, DriftMonitor,
    request_scope, current_context, StructuredLogger, TraceStore,
    SLO, burn_rate, AlertRule,
    assign_arm, two_proportion_ztest, GuardrailConfig, decide,
    FeedbackEvent, weak_label, GoldenSet, build_cases, FlywheelState,
)

rng = random.Random(SEED)
TOPICS = ["summarize", "extract_fields", "multi_table_math", "classify"]
# Champion is solid everywhere. The challenger looks fine on easy topics but
# regresses HARD on the two that matter - a realistic "shipped a bad prompt" bug.
CHAMP = {t: 0.95 for t in TOPICS}
CHAL  = {"summarize": 0.96, "classify": 0.96,
         "extract_fields": 0.50, "multi_table_math": 0.45}

trace_path = os.path.join(RUN_DIR, "spans.jsonl")
log_path   = os.path.join(RUN_DIR, "app.jsonl")
open(trace_path, "w").close()
logf = open(log_path, "w")

tracer = Tracer(trace_path)
logger = StructuredLogger(logf, context_provider=current_context)
drift  = DriftMonitor(baseline=0.92, window=20, alarm_drop=0.20)

# counters for the A/B guardrail (rollout pillar)
succ = {"A": 0, "B": 0}; n = {"A": 0, "B": 0}
events = []                      # FeedbackEvents for the flywheel pillar
# The rollout puts the challenger under scrutiny, so we alert on the CHALLENGER
# arm's rolling error rate (behind a 95% SLO) - the champion is our safe baseline.
slo = SLO("challenger_success", target=0.95, window_minutes=60)
alert = AlertRule("challenger_error_rate", fire_threshold=0.20,
                  recover_threshold=0.08, for_duration=3, recover_duration=3)
window_b, fired_minute = [], None

N = 600
for i in range(N):
    user = f"user_{i % 50:02d}"
    arm  = assign_arm(user, split=0.30, salt="obs-capstone")   # sticky bucketing
    topic = TOPICS[i % len(TOPICS)]
    p_ok = (CHAMP if arm == "A" else CHAL)[topic]
    ok = rng.random() < p_ok
    n_tokens = rng.randint(70, 160) if ok else rng.randint(5, 40)
    resp = {"n_tokens_out": n_tokens, "key_points": ["p"] if ok else []}
    if not ok:
        resp = {"error": "bad_output"} if rng.random() < 0.5 else resp

    # --- tracing: one span per attempt ---
    with tracer.span("task_attempt", topic=topic, arm=arm,
                     **{"gen_ai.request.model": "mock-1"}) as sp:
        sp.attrs["status"] = "ok" if ok else "error"

    # --- online_eval: label-free score + drift ---
    hs = heuristic_score(resp)["score"]
    drift_alarm = drift.update(hs)

    # --- logging_search: structured log inside a request scope ---
    with request_scope(user_id=user, session_id=f"sess_{i % 50:02d}") as ctx:
        if ok:
            logger.info("attempt_ok", topic=topic, arm=arm, latency_ms=n_tokens)
        else:
            logger.error("attempt_failed", topic=topic, arm=arm, latency_ms=n_tokens,
                         status="ERROR")

    # --- rollout: tally arm outcomes ---
    succ[arm] += int(ok); n[arm] += 1

    # --- alerting: challenger-arm rolling error rate -> AlertRule ---
    if arm == "B":
        window_b.append(0 if ok else 1); window_b = window_b[-20:]
        err_b = sum(window_b) / len(window_b)
        ev_alert = alert.update(i, err_b)
        if ev_alert and ev_alert.new_state.value == "firing" and fired_minute is None:
            fired_minute = i

    # --- feedback: build a signal-rich event for the flywheel ---
    events.append(FeedbackEvent(
        req_id=f"req_{i}", user_id=user, topic=topic,
        text=f"[{topic}] request {i} answer body",
        heuristic_score=hs, judge_score=(0.3 if not ok else 0.9),
        thumbs=(-1 if (not ok and rng.random() < 0.3) else None),
        regenerated=(not ok and rng.random() < 0.4),
        arm=("champion" if arm == "A" else "challenger")))

logf.flush(); logf.close()

# ---- pillar-by-pillar assertions -------------------------------------------
checks = {}

# tracing wrote one span line per attempt
spans = [json.loads(l) for l in open(trace_path) if l.strip()]
checks["tracing_wrote_spans"] = len(spans) == N and "trace_id" in spans[0]

# logging_search: the store finds failed attempts, and can slice by user
store = TraceStore(log_path)
failed = store.logs().failed()
checks["logs_found_failures"] = len(failed) > 0
u0_fail = store.logs().user("user_00").failed()
checks["logs_query_by_user"] = all(r["user_id"] == "user_00" for r in u0_fail)

# online_eval drift monitor ran on every attempt and holds a bounded rolling mean
import statistics as _stats
checks["drift_monitor_live"] = (len(drift.buf) == min(20, N)
                                and 0.0 <= _stats.mean(drift.buf) <= 1.0)

# alerting fired during the challenger meltdown
checks["alert_fired"] = fired_minute is not None

# rollout: the challenger is measurably worse; guardrail/decision says ROLLBACK
tr = two_proportion_ztest(succ["A"], n["A"], succ["B"], n["B"])
dec = decide(succ["A"], n["A"], succ["B"], n["B"], split=0.30,
             cfg=GuardrailConfig(slo=0.95, fast_burn=14.4, min_samples=150))
checks["challenger_worse"] = tr.diff < 0
checks["rollout_rolls_back"] = dec.action == "ROLLBACK"

# feedback: weak-label the traffic, mine failures into a fresh GoldenSet
golden = GoldenSet()
reviewed = [(e, "bad") for e in events if weak_label(e).label == "bad"][:40]
cases, stats = build_cases(reviewed, golden, source="production")
for c in cases:
    golden.add(c)
checks["flywheel_mined_cases"] = len(golden) > 0
checks["no_contamination"] = stats["already_in_golden"] == 0

flywheel = FlywheelState(turn=1, golden_size=len(golden),
                         human_labels_spent=len(reviewed))
flywheel.record(cases_added=len(cases))
checks["flywheel_recorded"] = len(flywheel.history) == 1

INTEGRATION_OK = all(checks.values())
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"\nchallenger arm success: A={succ['A']}/{n['A']}={succ['A']/n['A']:.3f}  "
      f"B={succ['B']}/{n['B']}={succ['B']/n['B']:.3f}")
print(f"rollout decision: {dec.action} ({dec.reason})")
print(f"golden set mined: {len(golden)} cases from {len(reviewed)} reviewed")
print(f"\nINTEGRATION_OK = {INTEGRATION_OK}")
assert INTEGRATION_OK, "the six pillars do NOT compose - a module drifted"

## 3. Build the wheel & check it

`python -m build` produces the sdist + wheel; `twine check` validates the
metadata that PyPI will read. Passing `twine check` is *necessary but not
sufficient* - it checks the packaging, not that the code works. That's why the
integration test above ran first, and why we do a fresh-wheel install next.

In [ ]:
# --- python -m build  +  twine check ----------------------------------------
dist = os.path.join(PKG_DIR, "dist")
shutil.rmtree(dist, ignore_errors=True)
b = subprocess.run([sys.executable, "-m", "build", PKG_DIR],
                   capture_output=True, text=True)
print(b.stdout[-400:])
if b.returncode != 0:
    print("BUILD STDERR:", b.stderr[-1000:])
artifacts = sorted(os.listdir(dist)) if os.path.isdir(dist) else []
print("dist/:", artifacts)
# pass explicit files to twine (no shell glob expansion in subprocess)
tw = subprocess.run([sys.executable, "-m", "twine", "check",
                     *[os.path.join(dist, a) for a in artifacts]],
                    capture_output=True, text=True)
print(tw.stdout.strip() or tw.stderr[-400:])
TWINE_OK = (b.returncode == 0 and tw.returncode == 0
            and any(a.endswith(".whl") for a in artifacts)
            and any(a.endswith(".tar.gz") for a in artifacts))
print("TWINE_OK =", TWINE_OK)
assert TWINE_OK

In [ ]:
# --- Fresh-install the built wheel into an ISOLATED dir and import from it ----
# This is the real "does the packaged artifact work?" test - separate from the
# source tree and the editable install, in its own clean environment.
whl = os.path.join(PKG_DIR, "dist",
                   [a for a in os.listdir(os.path.join(PKG_DIR, "dist"))
                    if a.endswith(".whl")][0])
target = os.path.join(BASE, "wheel_env")
shutil.rmtree(target, ignore_errors=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", target, whl],
               capture_output=True, text=True)
# import in a clean subprocess whose ONLY path to observability is `target`
probe = ("import sys; sys.path.insert(0, %r)\n"
         "import observability as o\n"
         "from observability import Tracer, decide, GoldenSet, AlertRule\n"
         "print(o.__version__)" % target)
res = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True,
                     cwd=BASE)
print("wheel:", os.path.basename(whl))
print("isolated import ->", res.stdout.strip() or res.stderr[-400:])
FRESH_OK = res.returncode == 0 and res.stdout.strip() == "0.1.0"
print("FRESH_OK =", FRESH_OK)
assert FRESH_OK

## 4. Launch day - your **third** public repo

You've done this twice (`paper-distiller` in Lesson 64, `agent-bench` in Lesson
70), so the mechanics are familiar. What's genuinely new at three repos is
**portfolio coherence**:

- **Timing gate.** Don't hard-launch two repos within ~1 week of each other -
  each launch deserves its own attention. `agent-obs` should trail `agent-bench`.
- **A real dependency story, not vanity links.** `agent-obs` isn't a random third
  project - it *instruments* `agent-bench` runs (you just proved it) and can watch
  a `paper-distiller` deployment in production. Each README links the siblings
  **because the code actually connects**, which is what makes a portfolio read as
  a system instead of a pile of repos.
- **Don't reuse the same "awesome-*" list** for all three submissions.

In [ ]:
# --- Write the launch/ops artifacts to disk ---------------------------------
# (README.md was already written as a package file in the consolidate step, so
#  the wheel build could read it. Here we add the ops artifacts around it.)
CHECKLIST = r"""# Launch checklist - agent-obs v0.1.0

## Portfolio gate (NEW at 3 repos)
- [ ] Last hard launch (agent-bench) was > 1 week ago
- [ ] README links agent-bench AND paper-distiller, links are live
- [ ] Cross-promo PR opened on agent-bench pointing back here
- [ ] Submitting to a DIFFERENT awesome-list than the previous two repos

## Ship
- [ ] `pytest` green locally
- [ ] `python -m build` && `twine check dist/*` pass
- [ ] Integration test (all six pillars in one run) passes in CI
- [ ] Tag only AFTER CI is green on main
- [ ] `git tag v0.1.0 && git push --tags`  (triggers OIDC publish)
- [ ] Verify `pip install agent-obs` in a clean venv
- [ ] Open 1-2 good-first-issues
"""

FIRST_ISSUE = r"""**good first issue: export spans to OpenTelemetry OTLP**

`Tracer` already uses OTel GenAI attribute names, so exporting to a real backend
(Honeycomb/Tempo/Langfuse) is small. Add `observability.exporters.otlp` that reads
the spans JSONL and POSTs OTLP over HTTP. Keep it optional (extras: `otlp`).

Good first issue because: the data model is done, it's additive, and it's easy to
test against a local collector. Ping in the PR if you want a design sketch.
"""

CI = r"""name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e . pytest build twine
      - run: pytest -q
      - run: python -m build
      - run: twine check dist/*
"""

RELEASE = r"""name: release
on:
  push:
    tags: ["v*"]
permissions:
  id-token: write        # OIDC Trusted Publishing - no long-lived PyPI token
  contents: read
jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install build && python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
"""

wf = os.path.join(PKG_DIR, ".github", "workflows")
os.makedirs(wf, exist_ok=True)
files = {
    os.path.join(PKG_DIR, "LAUNCH_CHECKLIST.md"): CHECKLIST,
    os.path.join(PKG_DIR, "first_issue.md"): FIRST_ISSUE,
    os.path.join(wf, "ci.yml"): CI,
    os.path.join(wf, "release.yml"): RELEASE,
}
for path, body in files.items():
    with open(path, "w") as f:
        f.write(body.lstrip("\n"))
    print("wrote", path.replace(BASE + "/", ""))

# assertions on the artifacts (README was written earlier as a package file)
readme_txt = open(os.path.join(PKG_DIR, "README.md")).read()
assert "agent-bench" in readme_txt and "paper-distiller" in readme_txt, "README must cross-promote both siblings"
assert "> 1 week" in CHECKLIST, "checklist must carry the portfolio timing gate"
assert "id-token: write" in RELEASE, "release must use OIDC Trusted Publishing"
print("\nlaunch/ops artifacts written & validated")

## 5. Launch runbook (yours to run & authorize)

Publishing under **your** GitHub/PyPI identity is yours to trigger - that's
correct by design, not a withheld capability. When you're ready:

```bash
cd agent-obs
git init && git add . && git commit -m "agent-obs v0.1.0"
gh repo create agent-obs --public --source=. --push
gh run watch                       # wait until CI is GREEN
python -m twine upload --repository testpypi dist/*   # dry run on TestPyPI
# register agent-obs as an OIDC Trusted Publisher in PyPI project settings
git tag v0.1.0 && git push --tags  # release.yml publishes via OIDC
pip install agent-obs              # verify in a clean venv
gh issue create -F first_issue.md  # seed a good-first-issue
```

Then open the cross-promo PR on `agent-bench` so the portfolio links both ways.

## 6. Capstone pitfalls

| Pitfall | Why it bites |
|---|---|
| Ship six modules with no integration test | they drift apart silently; users hit the seam |
| Assume `twine check` means "it works" | it only validates packaging metadata |
| Editable install passes, wheel is broken | test the built `.whl` in a *fresh* env too |
| Generic import name (`observability`) clashing on PyPI | keep distribution name (`agent-obs`) distinct and unique |
| Unresolved name collisions (`redact`, `burn_rate`) | decide the top-level winner explicitly in `__init__` |
| Tag before CI is green | you publish a broken release you can't unpublish |
| Long-lived PyPI token in CI | use OIDC Trusted Publishing instead |
| Two hard launches the same week | each launch starves the other of attention |
| README cross-promo links that 404 | portfolio reads as abandoned |
| Version string in two places | they drift; keep one `__version__` |

In [ ]:
# --- Verification checklist -------------------------------------------------
final = {}
final["package files on disk"]     = os.path.exists(os.path.join(OBS_DIR, "__init__.py"))
final["pyproject present"]         = os.path.exists(os.path.join(PKG_DIR, "pyproject.toml"))
final["integration composed"]      = INTEGRATION_OK
final["wheel built + twine ok"]    = TWINE_OK
final["fresh wheel install works"] = FRESH_OK
final["README cross-promotes"]     = os.path.exists(os.path.join(PKG_DIR, "README.md"))
final["CI workflow present"]       = os.path.exists(os.path.join(PKG_DIR, ".github/workflows/ci.yml"))
final["OIDC release present"]      = os.path.exists(os.path.join(PKG_DIR, ".github/workflows/release.yml"))
final["public API importable"]     = all(hasattr(obs, s) for s in
                                         ["Tracer", "decide", "GoldenSet", "AlertRule", "TraceStore"])

passed = sum(final.values())
for k, v in final.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print(f"\n{passed}/{len(final)} checks passed")
assert passed == len(final), "capstone incomplete"
print("\nagent-obs v0.1.0 is consolidated, tested end-to-end, built, and launch-ready.")

## 7. Summary, Phase 8 retrospective & what's next

**Today you learned** that a library is not a pile of tested modules - it's those
modules proven to *compose*, packaged behind one API, and shipped so a stranger
can `pip install` it. The integration test is the capstone's reason to exist.

### Phase 8 complete (Lessons 71-76)
| # | Lesson | Pillar shipped |
|---|---|---|
| 71 | Observability & online evals | `tracing`, `online_eval` |
| 72 | Structured logging & trace search | `logging_search` |
| 73 | Alerting, SLOs & on-call | `alerting` |
| 74 | A/B testing & guarded rollouts | `rollout` |
| 75 | Feedback loops & the data flywheel | `feedback` |
| 76 | **Capstone: ship `agent-obs`** | **the whole package** |

**You now own three cross-referencing open-source artifacts:** `paper-distiller`
(an agent), `agent-bench` (offline eval), and `agent-obs` (production ops) - a
believable end-to-end story for an AI engineer.

### Homework
1. Add `pytest` tests mirroring each in-notebook assertion, so CI enforces them.
2. Implement the OTLP exporter from `first_issue.md` and export a run to a local
   Tempo/Jaeger.
3. Wire `agent-obs` into the *real* `agent-bench` runner: emit one span + one
   structured log per task attempt, and fail CI if drift exceeds a threshold.
4. Add a `observability.dashboard` that renders metrics + drift as an HTML page.
5. Open the cross-promo PR on `agent-bench` and submit `agent-obs` to a new
   awesome-list.

### Phase 9 proposal (pick one; default = 9A)
- **9A - Advanced multi-agent orchestration** (default): planner/executor,
  supervisor trees, shared-memory blackboards, agent-to-agent protocols, and
  evaluating *multi*-agent systems (builds directly on agent-bench + agent-obs).
- **9B - RAG at production scale**: hybrid retrieval, re-ranking, eval-driven
  index tuning, freshness, and cost.
- **9C - Fine-tuning & post-training**: SFT, preference optimization (DPO), and
  when a fine-tune beats a better prompt.

*Reply with a topic or question and I'll adapt the curriculum; otherwise the next
scheduled run opens Phase 9A.*